# 23CSE301 Machine Learning — Capstone Project
## Classification Track — Review 1, Part A
**Team:** TASKMASTERS (Team 16)  
**Dataset:** Australian Open Tennis Matches (2008–2022)  
**Target:** `higher_rank_won`

This improved notebook keeps the five required Part A classifiers and adds clear sections, reusable evaluation, model comparison, confusion matrices, ROC curves, and cross-validation.

**Objective:** Predict whether the higher-ranked player won an Australian Open men's singles match using player-to-player differences. Place `AustralianOpen.csv` in `../data/` or the notebook's current folder.


## 1. Imports and reproducibility

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)


## 2. Load the dataset

In [ ]:
candidate_paths = [
    Path("../data/AustralianOpen.csv"),
    Path("data/AustralianOpen.csv"),
    Path("AustralianOpen.csv")
]
data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path, encoding="latin1")
    print(f"Loaded local dataset: {data_path}")
else:
    data_url = "https://data.scorenetwork.org/data/AustralianOpen.csv"
    df = pd.read_csv(data_url, encoding="latin1")
    print("Loaded dataset from SCORE data URL.")

print("Shape:", df.shape)
display(df.head())


## 3. Dataset audit and target distribution

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values (top 15 columns):")
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())

if "higher_rank_won" not in df.columns:
    raise KeyError("Expected target column 'higher_rank_won' was not found.")

target_counts = df["higher_rank_won"].astype("string").str.strip().str.title().value_counts(dropna=False)
display(target_counts.to_frame("match_count"))

target_counts_plot = target_counts[target_counts.index.notna()]
plt.figure(figsize=(6, 4))
target_counts_plot.plot(kind="bar")
plt.title("Target Distribution: Higher-Ranked Player Won?")
plt.xlabel("Target class")
plt.ylabel("Number of matches")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Feature engineering

The engineered features represent **absolute differences between the two players**:
- `rank_gap_abs`: absolute ranking difference
- `experience_gap_abs`: absolute difference in professional experience
- `height_gap_abs`: absolute height difference
- `weight_gap_abs`: absolute weight difference
- `same_handedness_bin`: Yes → 1, No → 0

The target is encoded as No → 0 and Yes → 1. Winner/loser names are not used as predictors.

In [ ]:
working_df = df.copy()
required = [
    "WRank", "LRank", "pl1_year_pro", "pl2_year_pro",
    "pl1_height", "pl2_height", "pl1_weight", "pl2_weight",
    "same_handedness", "higher_rank_won"
]
missing = [c for c in required if c not in working_df.columns]
if missing:
    raise KeyError(f"Required columns missing from CSV: {missing}")

working_df["rank_gap_abs"] = (working_df["WRank"] - working_df["LRank"]).abs()
working_df["experience_gap_abs"] = (working_df["pl1_year_pro"] - working_df["pl2_year_pro"]).abs()
working_df["height_gap_abs"] = (working_df["pl1_height"] - working_df["pl2_height"]).abs()
working_df["weight_gap_abs"] = (working_df["pl1_weight"] - working_df["pl2_weight"]).abs()
working_df["same_handedness_bin"] = (
    working_df["same_handedness"].astype("string").str.strip().str.title().map({"Yes": 1, "No": 0})
)
working_df["target"] = (
    working_df["higher_rank_won"].astype("string").str.strip().str.title().map({"No": 0, "Yes": 1})
)

feature_columns = [
    "rank_gap_abs", "experience_gap_abs", "height_gap_abs",
    "weight_gap_abs", "same_handedness_bin"
]
model_df = working_df[feature_columns + ["target"]].dropna(subset=["target"]).copy()
X = model_df[feature_columns]
y = model_df["target"].astype(int)

print("Rows used for modelling:", len(model_df))
print("Features:", feature_columns)
display(model_df[feature_columns + ["target"]].head())


## 5. EDA: feature distributions and correlation

In [ ]:
X.hist(figsize=(12, 8), bins=25)
plt.suptitle("Distributions of Engineered Classification Features", y=1.02)
plt.tight_layout()
plt.show()

corr_df = model_df[feature_columns + ["target"]].corr(numeric_only=True)
plt.figure(figsize=(9, 6))
plt.imshow(corr_df, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=45, ha="right")
plt.yticks(range(len(corr_df.index)), corr_df.index)
plt.title("Correlation Heatmap: Engineered Features and Target")
plt.tight_layout()
plt.show()


In [ ]:
scatter_df = model_df[["rank_gap_abs", "experience_gap_abs", "target"]].copy()
rng = np.random.default_rng(RANDOM_STATE)
scatter_df["target_jitter"] = scatter_df["target"] + rng.normal(0, 0.035, len(scatter_df))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(scatter_df["rank_gap_abs"], scatter_df["target_jitter"], alpha=0.35, s=18)
axes[0].set_title("Rank Gap vs Match Outcome")
axes[0].set_xlabel("Absolute rank gap")
axes[0].set_ylabel("Higher-ranked player won?")
axes[0].set_yticks([0, 1], labels=["No", "Yes"])

axes[1].scatter(scatter_df["experience_gap_abs"], scatter_df["target_jitter"], alpha=0.35, s=18)
axes[1].set_title("Experience Gap vs Match Outcome")
axes[1].set_xlabel("Absolute experience gap (years)")
axes[1].set_ylabel("Higher-ranked player won?")
axes[1].set_yticks([0, 1], labels=["No", "Yes"])
plt.tight_layout()
plt.show()


## 6. Train/test split and preprocessing

We use a stratified 80:20 split with `random_state=42`. Numerical missing values are imputed with the median. `StandardScaler` is used for Logistic Regression, KNN, Gaussian Naive Bayes and SVC; the Decision Tree uses imputation without scaling. Preprocessing is kept in pipelines to avoid fitting the imputer/scaler on the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaled_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), feature_columns)
])
tree_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), feature_columns)
])

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Training class proportions:")
display(y_train.value_counts(normalize=True).sort_index().to_frame("proportion"))


## 7. Build and tune the five required classifiers

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", scaled_preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "K-Nearest Neighbors": Pipeline([
        ("preprocess", scaled_preprocessor),
        ("model", KNeighborsClassifier())
    ]),
    "Gaussian Naive Bayes": Pipeline([
        ("preprocess", scaled_preprocessor),
        ("model", GaussianNB())
    ]),
    "Decision Tree": Pipeline([
        ("preprocess", tree_preprocessor),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "Support Vector Machine": Pipeline([
        ("preprocess", scaled_preprocessor),
        ("model", SVC(probability=True, random_state=RANDOM_STATE))
    ])
}

param_grids = {
    "K-Nearest Neighbors": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["minkowski", "manhattan"]
    },
    "Decision Tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [2, 3, 4, 5, 7, 10],
        "model__min_samples_leaf": [1, 2, 5]
    },
    "Support Vector Machine": {
        "model__C": [0.1, 1, 10, 100],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    }
}

fitted_models = {}
tuning_rows = []
for name, estimator in models.items():
    if name in param_grids:
        search = GridSearchCV(
            estimator, param_grids[name], cv=5,
            scoring="f1_weighted", n_jobs=-1, refit=True
        )
        search.fit(X_train, y_train)
        fitted_models[name] = search.best_estimator_
        tuning_rows.append({
            "Model": name,
            "Best CV weighted F1": search.best_score_,
            "Best parameters": search.best_params_
        })
    else:
        estimator.fit(X_train, y_train)
        fitted_models[name] = estimator

display(pd.DataFrame(tuning_rows))


## 8. Evaluate all classifiers on the same held-out test set

In [ ]:
results = []
test_predictions = {}
test_probabilities = {}

for name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    test_predictions[name] = y_pred
    test_probabilities[name] = y_prob
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })

classification_results_df = (
    pd.DataFrame(results).sort_values("F1_weighted", ascending=False).reset_index(drop=True)
)
display(classification_results_df.round(4))


In [ ]:
metric_columns = ["Accuracy", "Precision_weighted", "Recall_weighted", "F1_weighted", "ROC_AUC"]
plot_df = classification_results_df.set_index("Model")[metric_columns]
ax = plot_df.plot(kind="bar", figsize=(13, 6))
ax.set_title("Classification Model Comparison")
ax.set_xlabel("Classifier")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 9. Confusion matrices and classification reports

In [ ]:
for name, y_pred in test_predictions.items():
    print(f"\n{name} — classification report")
    print(classification_report(
        y_test, y_pred, labels=[0, 1],
        target_names=["Higher-ranked player did not win", "Higher-ranked player won"],
        zero_division=0
    ))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"])
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()


## 10. ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, y_prob in test_probabilities.items():
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
ax.set_title("ROC Curves — Classification Models")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 11. Decision Tree visualisation

In [ ]:
tree_model = fitted_models["Decision Tree"].named_steps["model"]
plt.figure(figsize=(20, 10))
plot_tree(
    tree_model, feature_names=feature_columns, class_names=["No", "Yes"],
    filled=True, rounded=True, max_depth=3, fontsize=8
)
plt.title("Decision Tree Classifier (first three levels shown)")
plt.tight_layout()
plt.show()


## 12. Five-fold cross-validation for the two strongest models

The two models are selected by their held-out test weighted F1 scores. Cross-validation is then run on the training set to estimate performance consistency across folds.

In [ ]:
top_two_models = classification_results_df.head(2)["Model"].tolist()
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name in top_two_models:
    scores = cross_val_score(
        fitted_models[name], X_train, y_train,
        cv=cv_splitter, scoring="f1_weighted", n_jobs=-1
    )
    cv_rows.append({
        "Model": name,
        "CV F1 mean": scores.mean(),
        "CV F1 std": scores.std()
    })

classification_cv_df = pd.DataFrame(cv_rows)
display(classification_cv_df.round(4))


## 13. Summary and GitHub notes

This notebook covers Logistic Regression, KNN, Gaussian Naive Bayes, Decision Tree, and SVC; EDA; feature engineering; stratified splitting; pipeline preprocessing; tuning; Accuracy, weighted Precision/Recall/F1, ROC-AUC; confusion matrices; ROC curves; tree visualisation; model comparison; and 5-fold cross-validation.

Run all cells top-to-bottom with the dataset available and inspect the actual results before writing conclusions into the README. Do not claim a winning model until the comparison table has been generated.

Suggested repository layout:
- `data/AustralianOpen.csv`
- `notebooks/classification.ipynb`
